# BudgetMem 7B (Qwen2.5) - Medium Docs Only (Focused, Resumable)

**Scope:** Three-way comparison on 200 medium-doc QA pairs at 7B (Qwen2.5-7B-Instruct).
- Baseline RAG (7B)
- BudgetMem (7B)
- LLMLingua-2 (BERT compression on CPU) + 7B generation

**Model:** Qwen2.5-7B-Instruct (open access, 4-bit quantization) - tests whether BudgetMem's
findings hold across model scale (3B to 7B) AND model family (Llama to Qwen).

**Resilience:**
- Every long cell saves progress every 10 iterations
- If Colab disconnects, just re-run the cell - it resumes from where it stopped
- Completed experiments are auto-skipped on re-run

**To run:** Restart runtime first (clears memory), select T4 GPU, run cells one at a time.

## 1. Setup

In [ ]:
!pip install -q transformers accelerate rank-bm25 nltk scikit-learn tqdm bitsandbytes llmlingua

import torch, numpy as np, json, re, string, time, random, os, gc
from datetime import datetime
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# Force UTF-8 to prevent ascii encoding errors
os.environ["PYTHONIOENCODING"] = "utf-8"
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
except Exception:
    pass

PROJECT_DIR = "/content/budgetmem_8b_focused"
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/checkpoints", exist_ok=True)

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")
print(f"Results dir: {PROJECT_DIR}/results/")

In [ ]:
# Qwen2.5-7B is OPEN ACCESS - no HuggingFace login needed!
# This cell is optional. You can skip it entirely.
# (Kept here only so cell numbering matches the instructions.)
print("Qwen2.5-7B is open access - no login required. Skipping.")

## 2. Utilities (same as 3B paper)

In [ ]:
def normalize_answer(s):
    s = (s or '').lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s

def compute_f1(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    if not pred_tokens or not truth_tokens:
        return float(pred_tokens == truth_tokens)
    common = set(pred_tokens) & set(truth_tokens)
    if not common:
        return 0.0
    prec = len(common) / len(pred_tokens)
    rec  = len(common) / len(truth_tokens)
    return 2 * prec * rec / (prec + rec)

def chunk_document(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunk = ' '.join(words[i:i + chunk_size])
        if len(chunk.split()) > 10:
            chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

def retrieve_bm25(query, chunks, top_k=3):
    if not chunks:
        return []
    tokenized = [c.split() for c in chunks]
    bm25 = BM25Okapi(tokenized)
    scores = bm25.get_scores(query.split())
    k = min(top_k, len(chunks))
    top_idx = np.argsort(scores)[-k:][::-1]
    return [chunks[i] for i in top_idx]

DISCOURSE_MARKERS = [
    'however','therefore','moreover','furthermore','consequently',
    'nevertheless','additionally','specifically','importantly',
    'in conclusion','on the other hand','as a result','for example',
    'in contrast','meanwhile','subsequently','nonetheless',
    'accordingly','hence','thus','indeed','notably',
    'in particular','conversely','alternatively','likewise',
    'similarly','in summary','to summarize','overall',
    'in other words','that is','namely','first','second',
    'third','finally','next','then','afterward',
    'before','after','during','while','although',
    'despite','regardless','provided that'
]

DEFAULT_WEIGHTS = {
    'entity_density':0.20,'tfidf_importance':0.20,
    'position_bias':0.15,'numerical_density':0.15,
    'discourse_markers':0.10,'question_presence':0.10,
}

def compute_features(chunks):
    n = len(chunks)
    if n == 0: return {}
    entity_raw = np.zeros(n)
    for i, c in enumerate(chunks):
        words = c.split()
        if words:
            entity_raw[i] = sum(1 for w in words if w and w[0].isupper()) / len(words)
    tfidf_raw = np.zeros(n)
    try:
        vec = TfidfVectorizer(max_features=500, stop_words='english')
        mat = vec.fit_transform(chunks)
        tfidf_raw = np.array(mat.mean(axis=1)).flatten()
    except Exception:
        pass
    position_raw = np.array([min(1.0, max(0.0, 1.3 - 2.0 * abs(i/n - 0.5))) for i in range(n)])
    number_raw = np.zeros(n)
    for i, c in enumerate(chunks):
        words = c.split()
        if words:
            number_raw[i] = sum(1 for w in words if re.search(r'\d', w)) / len(words)
    discourse_raw = np.zeros(n)
    for i, c in enumerate(chunks):
        lower_c = c.lower()
        words = c.split()
        discourse_raw[i] = sum(1 for m in DISCOURSE_MARKERS if m in lower_c) / max(len(words), 1)
    question_raw = np.zeros(n)
    interrogatives = {'who','what','where','when','why','how'}
    for i, c in enumerate(chunks):
        has_qmark = '?' in c
        has_interrog = bool(set(c.lower().split()) & interrogatives)
        question_raw[i] = min(1.0, (int(has_qmark) + int(has_interrog)) / 2.0)
    def norm(arr):
        return arr / arr.max() if arr.max() > 0 else arr
    return {
        'entity_density': norm(entity_raw),
        'tfidf_importance': norm(tfidf_raw),
        'position_bias': norm(position_raw),
        'numerical_density': norm(number_raw),
        'discourse_markers': norm(discourse_raw),
        'question_presence': norm(question_raw),
    }

def compute_salience(features_dict, weights=DEFAULT_WEIGHTS):
    n = len(next(iter(features_dict.values())))
    scores = np.zeros(n)
    for feat_name, feat_arr in features_dict.items():
        scores += weights.get(feat_name, 0) * feat_arr
    return scores

print("Utilities loaded.")

## 3. Generate medium-length structured papers (deterministic, same as 3B)

In [ ]:
def create_research_paper(idx):
    topics = ['machine learning','deep learning','neural networks','natural language processing',
              'computer vision','reinforcement learning','transfer learning','attention mechanisms',
              'graph neural networks','federated learning']
    methods = ['transformer','convolutional network','recurrent network',
               'generative adversarial network','variational autoencoder',
               'BERT','GPT','ResNet','diffusion model','mixture of experts']
    datasets_list = ['ImageNet','CIFAR-10','GLUE','SQuAD','COCO',
                     'WMT','CommonCrawl','WikiText-103','Penn Treebank','LibriSpeech']
    rng = random.Random(idx + 1000)
    topic  = topics[idx % len(topics)]
    method = methods[idx % len(methods)]
    ds     = datasets_list[idx % len(datasets_list)]
    acc    = round(85 + rng.uniform(0, 12), 1)
    prev   = round(acc - rng.uniform(2, 8), 1)
    lr     = round(rng.choice([1e-3, 3e-4, 5e-4, 1e-4]), 5)
    bs     = rng.choice([16, 32, 64, 128])
    epochs = rng.choice([50, 100, 200, 300])

    abstract = (f"This paper presents a novel approach to {topic} using a {method} architecture. "
                f"We evaluate on {ds} and achieve {acc}% accuracy, surpassing the previous "
                f"state-of-the-art result of {prev}%. Our method requires no task-specific "
                f"fine-tuning and trains in under 24 hours on a single GPU. "
                f"We release code and pretrained weights to support reproducibility.")
    intro = ((f"The field of {topic} has advanced rapidly. Classical approaches relied on hand-crafted "
              f"feature pipelines. Deep learning replaced these pipelines with end-to-end trainable models. "
              f"We propose a lightweight {method} variant. Our contributions are: (1) parameter-efficient "
              f"adaptation, (2) curriculum-based training, and (3) extensive evaluation on {ds} showing {acc}% accuracy. ") * 8)
    related = ((f"Prior work in {topic} spans several decades. Early statistical methods achieved moderate "
                f"success but struggled with real-world variability. Deep neural networks marked a turning point. "
                f"Recently, attention-based models have set new records. However, these models are computationally "
                f"expensive. Several groups have proposed efficiency improvements. ") * 7)
    methodology = ((f"Our proposed {method} variant introduces three modifications. First, we replace dense "
                    f"layers with sparse mixtures of experts. Second, we apply rotary position embeddings. "
                    f"Third, we use a staged training curriculum. Training uses AdamW with learning rate {lr}, "
                    f"batch size {bs}, for {epochs} epochs on 8 A100 GPUs. We apply gradient clipping at 1.0 "
                    f"and use cosine learning rate decay with a 5% warmup period. ") * 10)
    results = ((f"On {ds}, our model achieves {acc}% accuracy, compared to {prev}% for the previous best. "
                f"The improvement is statistically significant (p < 0.01). Ablation studies confirm each "
                f"modification contributes: removing sparse experts drops accuracy to {round(acc-3.2,1)}%, "
                f"removing rotary embeddings to {round(acc-1.8,1)}%, removing the curriculum to "
                f"{round(acc-2.5,1)}%. Training time is 18 hours. Inference latency is 12ms per example. ") * 7)
    discussion = ((f"Our results demonstrate that architectural efficiency and high accuracy are not "
                   f"mutually exclusive in {topic}. Limitations remain: our method has not been tested "
                   f"on languages other than English. Future work will explore multilingual settings. ") * 5)
    acknowledgements = (f"This work was supported by the National Science Foundation under grant "
                        f"IIS-{rng.randint(1800000,2100000)}. We thank the anonymous reviewers.")
    paper = (f"Title: A Novel Approach to {topic.title()} Using {method.title()}\n\n"
             f"Abstract: {abstract}\n\n1. Introduction\n{intro}\n\n2. Related Work\n{related}\n\n"
             f"3. Methodology\n{methodology}\n\n4. Results\n{results}\n\n5. Discussion\n{discussion}\n\n"
             f"Acknowledgements\n{acknowledgements}")
    qa_pairs = [
        (f"What accuracy does the proposed method achieve on {ds}?", f"{acc}%"),
        (f"What was the previous state-of-the-art accuracy?", f"{prev}%"),
        (f"What learning rate was used for training?", f"{lr}"),
        (f"How many epochs was the model trained for?", f"{epochs}"),
        (f"What batch size was used during training?", f"{bs}"),
    ]
    return paper, qa_pairs

medium_docs, medium_qa = [], []
for idx in range(40):
    paper, qas = create_research_paper(idx)
    for q, a in qas:
        medium_docs.append(paper)
        medium_qa.append({'question': q, 'answer': a})

print(f"Generated {len(medium_qa)} QA pairs from 40 papers.")
print(f"Avg doc length: {np.mean([len(d.split()) for d in medium_docs[::5]]):.0f} tokens")

## 4. Load Qwen2.5-7B (4-bit on T4)

Open-access 7B model, no HuggingFace gating. 4-bit quantization fits easily on T4.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # open access, no gating, 7B params

print("Loading Qwen2.5-7B model in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model.eval()
print(f"Model loaded. GPU usage: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── FIXED: use Qwen's proper chat template + terse system prompt ──
# This is the key fix. Last run used a raw prompt, so Qwen rambled and
# scored ~0.07. apply_chat_template() uses Qwen's ChatML format correctly,
# and the system message forces short factual answers (good for F1).
def generate_answer(context, question, max_ctx_chars=8000):
    messages = [
        {"role": "system", "content": "You are a precise question-answering assistant. "
         "Answer using ONLY the provided context. Give the shortest possible answer "
         "-- just the exact fact, number, name, or short phrase. Do not write full "
         "sentences. Do not explain."},
        {"role": "user", "content": f"Context: {context[:max_ctx_chars]}\n\nQuestion: {question}\n\nAnswer:"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=4096)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=40, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return answer.strip()

def baseline_answer(document, question):
    chunks = chunk_document(document)
    if not chunks: return generate_answer('', question)
    retrieved = retrieve_bm25(question, chunks, top_k=3)
    return generate_answer('\n\n'.join(retrieved), question)

def budgetmem_answer(document, question, budget_ratio=0.3):
    chunks = chunk_document(document)
    if not chunks: return generate_answer('', question)
    features = compute_features(chunks)
    salience = compute_salience(features)
    k = max(1, int(len(chunks) * budget_ratio))
    top_indices = np.argsort(salience)[-k:][::-1]
    selected = [chunks[i] for i in sorted(top_indices)]
    retrieved = retrieve_bm25(question, selected, top_k=3)
    return generate_answer('\n\n'.join(retrieved), question)

# Quick sanity check: answer one question to confirm the model isn't rambling
_test = generate_answer("The model achieves 92.5% accuracy on ImageNet.", "What accuracy is achieved on ImageNet?")
print(f"\nSANITY CHECK -- should be short like '92.5%':")
print(f"  Model answered: '{_test}'")
print("\nIf that answer is short and correct, the chat template fix works. Proceed.")
print("If it's a long rambling sentence, STOP and tell me.")

## 5. Resumable runner (handles disconnects + auto-skip)

In [ ]:
def run_resumable(method_name, method_fn, docs, qa_pairs, results_dir):
    """
    Resumable runner: saves progress every 10 iterations,
    skips if final result file already exists.
    """
    final_file = f"{results_dir}/{method_name}_final.json"
    partial_file = f"{results_dir}/{method_name}_partial.json"

    # Skip if already done
    if os.path.exists(final_file):
        with open(final_file) as f:
            data = json.load(f)
        print(f"[SKIP] {method_name} already complete. F1 = {data['f1_mean']:.4f}")
        return data['f1_mean'], data.get('f1s', [])

    # Resume from partial if exists
    if os.path.exists(partial_file):
        with open(partial_file) as f:
            saved = json.load(f)
        f1s = saved['f1s']
        start_idx = len(f1s)
        print(f"[RESUME] {method_name}: continuing from {start_idx}/{len(qa_pairs)}")
    else:
        f1s = []
        start_idx = 0
        print(f"[START] {method_name}: fresh run")

    torch.manual_seed(42)
    pbar = tqdm(range(start_idx, len(qa_pairs)), desc=method_name,
                initial=start_idx, total=len(qa_pairs))
    for i in pbar:
        try:
            pred = method_fn(docs[i], qa_pairs[i]['question'])
            f1 = compute_f1(pred, qa_pairs[i]['answer'])
        except Exception as e:
            print(f"\nError at {i}: {e}")
            f1 = 0.0
        f1s.append(f1)

        # Save every 10 iterations
        if (i + 1) % 10 == 0:
            with open(partial_file, 'w') as f:
                json.dump({'f1s': f1s, 'completed': i + 1, 'method': method_name}, f)

    # Final save
    mean_f1 = float(np.mean(f1s))
    with open(final_file, 'w') as f:
        json.dump({
            'method': method_name,
            'f1_mean': mean_f1,
            'f1s': f1s,
            'n_examples': len(qa_pairs),
            'timestamp': datetime.now().isoformat()
        }, f)
    # Remove partial after final save
    if os.path.exists(partial_file):
        os.remove(partial_file)

    print(f"\n[DONE] {method_name}: F1 = {mean_f1:.4f}")
    return mean_f1, f1s

print("Resumable runner defined.")

## 6. Run Baseline 7B (~40 min)

If this crashes or disconnects, just re-run the cell — it resumes from where it stopped.

In [ ]:
baseline_f1, baseline_scores = run_resumable(
    "baseline_8b", baseline_answer, medium_docs, medium_qa,
    f"{PROJECT_DIR}/results"
)

## 7. Run BudgetMem 7B (~60 min)

Same resilience — re-run if interrupted.

In [ ]:
budgetmem_f1, budgetmem_scores = run_resumable(
    "budgetmem_8b", budgetmem_answer, medium_docs, medium_qa,
    f"{PROJECT_DIR}/results"
)

## 8. Pre-compress documents with LLMLingua-2 (CPU, ~20 min)

LLMLingua-2 runs on CPU to avoid OOM (Qwen2.5-7B stays on GPU).
Compresses all 200 unique docs and saves to disk. Qwen2.5-7B then generates from compressed text.

In [ ]:
compressed_file = f"{PROJECT_DIR}/results/llmlingua_compressed.json"
compressed_partial = f"{PROJECT_DIR}/results/llmlingua_compressed_partial.json"

if os.path.exists(compressed_file):
    with open(compressed_file) as f:
        compressed_docs = json.load(f)['compressed']
    print(f"[SKIP] Already compressed {len(compressed_docs)} docs.")
else:
    # Resume partial if exists
    if os.path.exists(compressed_partial):
        with open(compressed_partial) as f:
            compressed_docs = json.load(f)['compressed']
        print(f"[RESUME] Continuing from {len(compressed_docs)} compressed docs.")
    else:
        compressed_docs = []
        print("[START] Loading LLMLingua-2 on CPU...")

    from llmlingua import PromptCompressor
    print("Loading LLMLingua-2 (CPU mode to avoid OOM)...")
    llm_lingua = PromptCompressor(
        model_name="microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank",
        use_llmlingua2=True,
        device_map="cpu"
    )
    print("LLMLingua-2 loaded on CPU.")

    # Compress each unique document
    for i in tqdm(range(len(compressed_docs), len(medium_docs)), desc="Compressing",
                  initial=len(compressed_docs), total=len(medium_docs)):
        try:
            result = llm_lingua.compress_prompt(
                medium_docs[i], rate=0.3,
                force_tokens=['?','.','!',','], drop_consecutive=True
            )
            compressed_docs.append(result['compressed_prompt'])
        except Exception as e:
            print(f"\nError compressing {i}: {e}")
            compressed_docs.append(medium_docs[i][:2000])  # fallback: truncate

        if (i + 1) % 10 == 0:
            with open(compressed_partial, 'w') as f:
                json.dump({'compressed': compressed_docs, 'completed': i + 1}, f)

    # Save final and clean up
    with open(compressed_file, 'w') as f:
        json.dump({'compressed': compressed_docs, 'n': len(compressed_docs)}, f)
    if os.path.exists(compressed_partial):
        os.remove(compressed_partial)

    # Free LLMLingua from memory
    del llm_lingua
    gc.collect()
    print(f"\nCompressed {len(compressed_docs)} docs. LLMLingua freed from memory.")

## 9. Run LLMLingua-2 generation pass (~40 min)

Uses pre-compressed text + Qwen2.5-7B for answers.

In [ ]:
def llmlingua_answer(idx, question):
    """Use pre-compressed text from disk + BM25 + Qwen2.5-7B."""
    compressed_text = compressed_docs[idx]
    chunks = chunk_document(compressed_text)
    if not chunks:
        chunks = [compressed_text[:1000]]
    retrieved = retrieve_bm25(question, chunks, top_k=3)
    return generate_answer('\n\n'.join(retrieved), question)

# Wrapper to match signature
def llmlingua_method(doc, question):
    idx = medium_docs.index(doc)
    return llmlingua_answer(idx, question)

# Actually faster: use index directly via a parallel docs list
# Build (idx, question, answer) mapping
indexed_qa = [(i, medium_qa[i]) for i in range(len(medium_qa))]

# Custom resumable run for LLMLingua (uses doc index, not doc text)
ll_final = f"{PROJECT_DIR}/results/llmlingua_8b_final.json"
ll_partial = f"{PROJECT_DIR}/results/llmlingua_8b_partial.json"

if os.path.exists(ll_final):
    with open(ll_final) as f:
        ll_data = json.load(f)
    llmlingua_f1 = ll_data['f1_mean']
    print(f"[SKIP] LLMLingua-2 7B already complete. F1 = {llmlingua_f1:.4f}")
else:
    if os.path.exists(ll_partial):
        with open(ll_partial) as f:
            saved = json.load(f)
        ll_f1s = saved['f1s']
        start = len(ll_f1s)
        print(f"[RESUME] LLMLingua-2 7B from {start}/{len(medium_qa)}")
    else:
        ll_f1s = []
        start = 0

    torch.manual_seed(42)
    for i in tqdm(range(start, len(medium_qa)), desc="LLMLingua-2 7B",
                  initial=start, total=len(medium_qa)):
        try:
            pred = llmlingua_answer(i, medium_qa[i]['question'])
            f1 = compute_f1(pred, medium_qa[i]['answer'])
        except Exception as e:
            print(f"Error at {i}: {e}")
            f1 = 0.0
        ll_f1s.append(f1)

        if (i + 1) % 10 == 0:
            with open(ll_partial, 'w') as f:
                json.dump({'f1s': ll_f1s, 'completed': i + 1}, f)

    llmlingua_f1 = float(np.mean(ll_f1s))
    with open(ll_final, 'w') as f:
        json.dump({
            'method': 'llmlingua_8b',
            'f1_mean': llmlingua_f1,
            'f1s': ll_f1s,
            'n_examples': len(medium_qa)
        }, f)
    if os.path.exists(ll_partial):
        os.remove(ll_partial)
    print(f"\n[DONE] LLMLingua-2 7B: F1 = {llmlingua_f1:.4f}")

## 10. Final Summary

In [ ]:
print("=" * 65)
print("BUDGETMEM 7B MEDIUM-DOC RESULTS (T4, 4-bit)")
print("=" * 65)
print(f"{'Method':<20} {'7B F1':<12} {'3B F1 (paper)':<15} {'Note':<20}")
print("-" * 65)
print(f"{'Baseline RAG':<20} {baseline_f1:<12.4f} {0.855:<15.4f} {'all chunks':<20}")
print(f"{'LLMLingua-2':<20} {llmlingua_f1:<12.4f} {0.532:<15.4f} {'token compress':<20}")
print(f"{'BudgetMem':<20} {budgetmem_f1:<12.4f} {0.859:<15.4f} {'chunk select':<20}")
print("-" * 65)

# Key comparison
print("\nKey questions for paper:")
if budgetmem_f1 > llmlingua_f1:
    print(f"  - BudgetMem ({budgetmem_f1:.4f}) > LLMLingua-2 ({llmlingua_f1:.4f}): CONFIRMED at 7B")
else:
    print(f"  - BudgetMem ({budgetmem_f1:.4f}) <= LLMLingua-2 ({llmlingua_f1:.4f}): ranking REVERSED at 7B")

gap_vs_base = abs(budgetmem_f1 - baseline_f1) / baseline_f1 * 100 if baseline_f1 > 0 else 0
print(f"  - BudgetMem gap vs Baseline: {gap_vs_base:.1f}%")

# Save combined
summary = {
    'experiment': 'BudgetMem 7B - Medium Docs Three-Way',
    'model': 'Qwen/Qwen2.5-7B-Instruct',
    'precision': '4-bit NF4 (bitsandbytes)',
    'gpu': torch.cuda.get_device_name(0),
    'n_examples': len(medium_qa),
    'results': {
        'baseline_rag': float(baseline_f1),
        'llmlingua_2': float(llmlingua_f1),
        'budgetmem':   float(budgetmem_f1),
    },
    'reference_3b': {
        'baseline_rag': 0.855,
        'llmlingua_2': 0.532,
        'budgetmem': 0.859,
    },
    'timestamp': datetime.now().isoformat()
}
with open(f"{PROJECT_DIR}/results/FINAL_7B_MEDIUM_SUMMARY.json", 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nFinal summary saved to:")
print(f"  {PROJECT_DIR}/results/FINAL_7B_MEDIUM_SUMMARY.json")
print("\nDownload that file from the Colab file browser and share back.")